# 1: EchoNext data and released Mini baseline

Attach the team's private EchoNext dataset before running. It must contain the official filenames, extracted from any archive. Enable Internet for repository/dependency/model downloads. A CPU session is enough for the first run; CUDA is optional.

This notebook imports the shared Python implementation. It has no embedded patient data or saved outputs. Keep notebook versions and generated patient-level files private. See `docs/data_and_baseline.md` for the data contract and handoff to B/C/E.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/kaggle/working/compress_ECG')
BRANCH = 'shaurya/data-loader-and-baselines'
# Set to a reviewed commit SHA to repeat an exact code version.
CODE_COMMIT = None
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH,
                    'https://github.com/andykimdev/compress_ECG.git', str(REPO_DIR)], check=True)
if CODE_COMMIT:
    subprocess.run(['git', 'checkout', CODE_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)], check=True)
# Allow imports in this running kernel immediately after the editable install.
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Code commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip())


## Locate the attached files
Set `DATA_DIR` explicitly if you have more than one copy attached. It must be the directory containing `EchoNext_metadata_100k.csv` and the split arrays, even if the Kaggle upload added nested folders.


In [ ]:
DATA_DIR = None  # Example: Path('/kaggle/input/your-private-dataset/echonext/1.1.1')
if DATA_DIR is None:
    matches = list(Path('/kaggle/input').rglob('EchoNext_metadata_100k.csv'))
    if len(matches) != 1:
        raise ValueError('Set DATA_DIR to the exact dataset root; found %d metadata files' % len(matches))
    DATA_DIR = matches[0].parent
DATA_DIR = Path(DATA_DIR)
print('Dataset root:', DATA_DIR)


## Check a batch
Waveforms are `[batch, 12, 2500]` at 250 Hz. Tabular inputs are the seven supplied preprocessed features. Labels use the released 12-output order; `label_mask` identifies observed targets. Never turn missing component targets into negative training examples. `no_split` is excluded.


In [ ]:
import torch
from torch.utils.data import DataLoader
from compress_ecg.data import EchoNextDataset, LABEL_COLUMNS

dataset = EchoNextDataset(DATA_DIR, 'val', limit=1000, seed=42)
batch = next(iter(DataLoader(dataset, batch_size=8, shuffle=False)))
print({name: tuple(batch[name].shape) for name in ('waveform', 'tabular', 'labels', 'label_mask')})
print('Selected validation ECGs:', len(dataset))
print('Observed targets per output:', batch['label_mask'].sum(0).tolist())
print('Label order:', LABEL_COLUMNS)
# B/C: use split='train' and a shuffled DataLoader for training.
# All model losses must respect label_mask; this loader does not resample to 500 Hz.


## Download Mini and run the baseline
The downloader checks fixed SHA-256 hashes and skips files already verified. This mode downloads only Mini, not the teacher weights. Run on 1,000 seeded validation ECGs first. The 0.820 paper result is a full test-set reference, not a target for this validation subset.


In [ ]:
subprocess.run(['bash', 'scripts/download_weights.sh', '--mini-only'], cwd=REPO_DIR, check=True)

from compress_ecg.baseline import run_baseline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/runs/mini_val_1000_seed42')
# A completed run is not overwritten. Choose a fresh OUTPUT_DIR for another run.
result = run_baseline(
    DATA_DIR, OUTPUT_DIR, split='val', limit=1000, seed=42,
    batch_size=64, num_workers=0, device=DEVICE,
    mini_dir=REPO_DIR / 'external/IntroECG/7-EchoNext Minimodel',
    data_version='1.1.1',
)
print('Saved:', result['output_dir'])
print('ECGs:', result['n_records'])
print('SHD metrics:', result['metrics'][LABEL_COLUMNS[-1]])


## Handoff to E
The private output folder contains `predictions.npz`, `metadata.csv`, `manifest.json`, and `metrics.json`. Save this notebook's private output version so E can attach it as an input. Predictions and metadata are in identical row order and can also be joined on `row_id`.

The manifest records code commit/dirty state, dataset version, metadata hash, split, seed, sample count, input shapes, and model hashes. Keep the original Kaggle dataset version identifier with your experiment notes as well. Only reviewed aggregate metrics and figures belong in GitHub.


In [ ]:
import numpy as np
import pandas as pd

with np.load(OUTPUT_DIR / 'predictions.npz', allow_pickle=False) as saved:
    probabilities = saved['probabilities']
    labels = saved['labels']  # NaN for unobserved components
    label_mask = saved['label_mask']
    row_ids = saved['row_id']
audit = pd.read_csv(OUTPUT_DIR / 'metadata.csv', dtype={'patient_key': 'string'})
assert np.array_equal(row_ids, audit['row_id'].to_numpy())
print('Prediction shape:', probabilities.shape)
print('All finite:', bool(np.isfinite(probabilities).all()))
# E computes calibration plots and subgroup performance from these aligned files.
